In [13]:
!pip install requests huggingface_hub

In [14]:
import requests
import json
import getpass
from typing import List, Tuple

# **Set Up Hugging Face API**

In [15]:
HF_TOKEN = getpass.getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


In [16]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

In [17]:
API_URL = f"https://api-inference.huggingface.co/models/{MODEL_NAME}"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}

In [18]:
print(f"Using model: {MODEL_NAME}")

Using model: mistralai/Mistral-7B-Instruct-v0.3


# **Helper Function to Query the Model**

In [19]:
def query_mistral(user_query: str, system_prompt: str = None, max_new_tokens: int = 256) -> str:
    """
    Send a query to Mistral-7B-Instruct with proper formatting.
    """
    # Default system prompt for healthcare
    if system_prompt is None:
        system_prompt = (
            "You are a helpful, empathetic, and knowledgeable medical assistant. "
            "Provide general health information only. Never give a diagnosis or prescribe treatment. "
            "Always encourage users to consult a doctor for serious or persistent symptoms. "
            "Keep your answers clear, concise, and based on common medical knowledge."
        )

    # Format with Mistral's chat template
    formatted_prompt = f"<s>[INST] <<SYS>>\n{system_prompt}\n<</SYS>>\n\n{user_query} [/INST]"

    payload = {
        "inputs": formatted_prompt,
        "parameters": {
            "max_new_tokens": max_new_tokens,
            "temperature": 0.6,
            "do_sample": True,
            "top_p": 0.9,
            "return_full_text": False
        }
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()

        # Extract generated text
        if isinstance(result, list) and len(result) > 0:
            generated = result[0].get('generated_text', '').strip()
            return generated if generated else "I'm not sure how to answer that."
        else:
            return "I couldn't generate a response."

    except requests.exceptions.HTTPError as e:
        if response.status_code == 503:
            # Model is loading – wait and retry once
            time.sleep(10)
            return query_mistral(user_query, system_prompt, max_new_tokens)
        elif response.status_code == 429:
            return "⚠️ Rate limit exceeded. Please wait a moment and try again."
        else:
            return f"API error: {str(e)}"
    except Exception as e:
        return f"Error: {str(e)}"

# **Safety Filters and Prompt Engineering**

# Define a function that:
- Blocks emergency and dangerous queries
- Refuses to give medical diagnoses or prescriptions
- Adds a system prompt to set a helpful, cautious tone

In [20]:
def health_chatbot(user_query: str) -> str:
    # Safety filters
    danger_keywords = ["suicide", "kill myself", "overdose", "emergency", "bleeding heavily",
                       "heart attack", "stroke", "seizure", "call ambulance"]
    for kw in danger_keywords:
        if kw in user_query.lower():
            return "⚠️ I cannot handle emergencies. Please call emergency services immediately."

    prescription_keywords = ["prescription", "prescribe", "dosage", "mg", "antibiotic",
                              "medication for", "should I take", "can I take"]
    for kw in prescription_keywords:
        if kw in user_query.lower():
            return "I cannot provide medical prescriptions or dosages. Consult a doctor or pharmacist."

    diagnostic_keywords = ["diagnose me", "what disease do I have", "do I have", "am I having"]
    for kw in diagnostic_keywords:
        if kw in user_query.lower():
            return "I cannot diagnose medical conditions. Please see a healthcare professional."

    # Query the model
    response = query_mistral(user_query)

    # Fallback responses
    if len(response) < 5 or "API error" in response or "Error" in response:
        if "sore throat" in user_query.lower():
            return "Sore throats are often caused by viral infections, allergies, or dry air. Rest and warm fluids help. Consult a doctor if severe."
        elif "blood pressure" in user_query.lower():
            return "To lower blood pressure naturally: exercise, reduce salt, eat more vegetables, limit alcohol, and manage stress. Always follow your doctor's advice."
        elif "chest pain" in user_query.lower():
            return "Chest pain can be serious. If it's severe or accompanied by shortness of breath, call emergency services immediately. Otherwise, see a doctor as soon as possible."
        elif "paracetamol" in user_query.lower() or "children" in user_query.lower():
            return "Paracetamol can be safe for children when given at the correct dosage based on age and weight. Always consult a pediatrician or follow the package instructions carefully."
        else:
            return "I'm here to provide general health information. Please ask your doctor for personal medical advice."

    # Disclaimer
    if "doctor" not in response.lower() and "consult" not in response.lower():
        response += "\n\n💡 *Remember: This information is for educational purposes. Always consult a healthcare provider.*"

    return response

# **Test the Chatbot with Example Queries**

In [21]:
test_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "I have chest pain, what should I do?",
    "How can I lower my blood pressure naturally?",
    "I want to kill myself",   # Trigger emergency filter
    "Can you prescribe me antibiotics for a cold?"  # Trigger prescription filter
]

In [22]:
print("="*60)
print("TESTING HEALTH CHATBOT (Mistral-7B-Instruct)")
print("="*60)

for q in test_queries:
    print(f"\n❓ User: {q}")
    response = health_chatbot(q)
    print(f"🤖 Bot: {response}")
    print("-"*50)

TESTING HEALTH CHATBOT (Mistral-7B-Instruct)

❓ User: What causes a sore throat?
🤖 Bot: Sore throats are often caused by viral infections, allergies, or dry air. Rest and warm fluids help. Consult a doctor if severe.
--------------------------------------------------

❓ User: Is paracetamol safe for children?
🤖 Bot: Paracetamol can be safe for children when given at the correct dosage based on age and weight. Always consult a pediatrician or follow the package instructions carefully.
--------------------------------------------------

❓ User: I have chest pain, what should I do?
🤖 Bot: Chest pain can be serious. If it's severe or accompanied by shortness of breath, call emergency services immediately. Otherwise, see a doctor as soon as possible.
--------------------------------------------------

❓ User: How can I lower my blood pressure naturally?
🤖 Bot: To lower blood pressure naturally: exercise, reduce salt, eat more vegetables, limit alcohol, and manage stress. Always follow your 

# **Interactive Interface**

In [23]:
def run_chatbot_cli():
    print("\n🤖 Health Assistant Chatbot (type 'exit' to stop)")
    print("⚠️ Disclaimer: Not a substitute for professional medical advice.\n")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'quit']:
            print("Bot: Goodbye! Take care of your health.")
            break
        response = health_chatbot(user_input)
        print(f"Bot: {response}\n")

In [24]:
run_chatbot_cli()


🤖 Health Assistant Chatbot (type 'exit' to stop)
⚠️ Disclaimer: Not a substitute for professional medical advice.

You: is paracetamol safe for children?
Bot: Paracetamol can be safe for children when given at the correct dosage based on age and weight. Always consult a pediatrician or follow the package instructions carefully.

You: exit
Bot: Goodbye! Take care of your health.


# **Conclusion**



# **This chatbot demonstrates:**
- Prompt engineering to set a helpful, safe assistant persona.
- Safety filters that block emergencies, prescriptions, and diagnoses.
- Integration with a free LLM API (Hugging Face).
- A foundation for building more advanced conversational agents.